## Step 1 - Prepare data
Raw data needs to be imported with valid rays (i.e. rays that pass through the designated plot area) saved to parquet formats. The resulting formatting will save all rays with the following dataframe structure:

ray_id | origin (x, y, z) | direction (x, y, z) | points (x, y, z)

Unbound rays (i.e. no hit) will have an empty entry in points column. This approach is more perfomative but is less storage efficient.

### Importing legs from Helios simulations
Use the following section of the notebook to convert helios outputs (namely *pulses.txt and *points.xyz) into the appropriate format

In [13]:
import xarray as xr
import dask
import dask.array as da
import dask.dataframe as dd
from dask.diagnostics import ProgressBar
import pyarrow as pa
import pandas as pd
import numpy as np
import os
import shutil

In [14]:
# Setup the project

# Define the output directory
base_output_dir = "result_analysis/LAD_Estimations_PointCloud"
os.makedirs(base_output_dir, exist_ok=True)

# Define log file path
log_file = "lad_estimation_pipeline.log"

# # Create a multiprocessing queue for logging
# log_queue = mp.Queue(-1)
# listener = mp.Process(target=listener_process, args=(log_queue, listener_configurer, log_file))
# listener.start()

# # Configure worker loggers
# worker_configurer(log_queue)
# logger = get_logger(__name__)

# Define file templates
voxel_centers_template = "200_leaf_60_voxel_size_{voxel_size}.csv"
xyz_template = "200_Leaf_60/leg{leg:03d}_points.xyz"
pulses_template = "200_Leaf_60/leg{leg:03d}_pulse.txt"
fullwave_template = "200_Leaf_60/leg{leg:03d}_fullwave.txt"
valid_rays_template = "200_Leaf_60/leg_{leg:d}_valid_rays.parquet"

# Define selected legs
selected_legs = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]

# Define voxel sizes
voxel_sizes = [0.5]

# Define Parquet schemas
# Save non-hit rays to .parquet
valid_rays_schema = pa.schema([
    ('ray_id', pa.int64()),
    ('origin_x', pa.float32()),
    ('origin_y', pa.float32()),
    ('origin_z', pa.float32()),
    ('direction_x', pa.float32()),
    ('direction_y', pa.float32()),
    ('direction_z', pa.float32()),
    ('points_x', pa.float32()),
    ('points_y', pa.float32()),
    ('points_z', pa.float32())
])
    
voxel_centers_schema = pa.schema({
    'voxel_id': pa.int64(),
    'voxel_center_x': pa.float32(),
    'voxel_center_y': pa.float32(),
    'voxel_center_z': pa.float32(),
    'voxel_min_x': pa.float32(),
    'voxel_min_y': pa.float32(),
    'voxel_min_z': pa.float32(),
    'voxel_max_x': pa.float32(),
    'voxel_max_y': pa.float32(),
    'voxel_max_z': pa.float32()
})

In [ ]:
# Setup the voxel dataframes
voxels_dict = {}
plot_mins = []
plot_maxs = []

for voxel_size in voxel_sizes:
    # Extract voxel centres from file
    voxel_centers_file = voxel_centers_template.format(voxel_size=voxel_size)
    voxel_centers = dd.read_csv(
        voxel_centers_file, 
        delimiter=',', 
        names=[
            'x', 
            'y', 
            'z', 
            'angle', 
            'G_mean', 
            'mean_angle', 
            'clumping_index', 
            'plant_area', 
            'leaf_area_index', 
            'total_surface_area', 
            'total_number_of_rays', 
            'clipped_hit_count',
            'voxel_hit_count',
            'mean_path_length',
            'sum_path_length',
            'mean_free_path_length',
            'sum_free_path_length',
            'mean_effective_free_path_length',
            'sum_effective_free_path_length',
            'var_effective_path_length',
            'mean_effective_path_length',
            'sum_hits_ze',
            'LAD_BeerLambert',
            'LAD_MLE_MCF',
            'LAD_BiasCorrectedBeerLambert',
            'LAD_BiasCorrectedUnequalPathLengths',
            'LAD_TBC_MLE'
        ], 
        header=0
        )
    

    # Find the smallest x, y, z values
    # voxel_radius = voxel_size/2
    # voxel_centers['min_x'] = voxel_centers['x'].min().compute() - voxel_radius
    # voxel_centers['min_y'] = voxel_centers['y'].min().compute() - voxel_radius
    # voxel_centers['min_z'] = voxel_centers['z'].min().compute() - voxel_radius
    # # plot_mins.append([voxel_centers['min_x'], voxel_centers['min_y'], voxel_centers['min_z']])
    # plot_mins.append([voxel_centers['min_x'].min().compute(), voxel_centers['min_y'].min().compute(), voxel_centers['min_z'].min().compute()])
    
    # # Find the largest x, y, z values
    # voxel_centers['max_x'] = voxel_centers['x'].max().compute() + voxel_radius
    # voxel_centers['max_y'] = voxel_centers['y'].max().compute() + voxel_radius
    # voxel_centers['max_z'] = voxel_centers['z'].max().compute() + voxel_radius
    # plot_maxs.append([voxel_centers['max_x'].max().compute(),  voxel_centers['max_y'].max().compute(), voxel_centers['max_z'].max().compute()])

    voxel_centers = np.unique(voxel_centers[['x', 'y', 'z']], axis=0)
    voxel_id = np.arange(len(voxel_centers)) + int(voxel_size * 1000) * 1000000
    voxel_radius = voxel_size / 2
    voxel_centers = np.hstack([
        voxel_id.reshape(-1, 1),
        voxel_centers,
        voxel_centers - voxel_radius,
        voxel_centers + voxel_radius
    ])
    tolerance = 0.1
    plot_mins.append(voxel_centers[:, 4:7].min(axis=0) - tolerance)
    plot_maxs.append(voxel_centers[:, 7:10].max(axis=0) + tolerance)

    # Convert voxel_centers to dask dataframe
    voxel_centers_dd = dd.from_array(voxel_centers, columns=['voxel_id', 'voxel_center_x', 'voxel_center_y', 'voxel_center_z', 'voxel_min_x', 'voxel_min_y', 'voxel_min_z', 'voxel_max_x', 'voxel_max_y', 'voxel_max_z'])
    # voxel_centers_dd = voxel_centers_dd.repartition(npartitions=3)

    # Add to dictionary
    voxels_dict[voxel_size] = voxel_centers

# Establish plot boundaries
plot_min = np.min(plot_mins, axis=0)
plot_max = np.max(plot_maxs, axis=0)

# Setup lists of dask dataframes
rays_list = []

# Function for traversing voxels
def traverse_voxels(ray_origins, ray_directions, voxel_min, voxel_max):
    t_min = da.divide(voxel_min - ray_origins, ray_directions, out=da.full_like(ray_origins, np.inf), where=da.all(ray_directions!=0))
    t_max = da.divide(voxel_max - ray_origins, ray_directions, out=da.full_like(ray_origins, np.inf), where=da.all(ray_directions!=0))
    t1 = da.minimum(t_min, t_max)
    t2 = da.maximum(t_min, t_max)
    t_enter = da.max(t1, axis=1)
    t_exit = da.min(t2, axis=1)
    mask = (t_enter <= t_exit) & (t_exit >= 0)
    return mask

def valid_mask(df):
    origins = da.array([df['origin_x'].values, df['origin_y'].values, df['origin_z'].values]).T
    directions = da.array([df['direction_x'].values, df['direction_y'].values, df['direction_z'].values]).T
    mask = traverse_voxels(origins, directions, plot_min, plot_max).compute()
    return df[mask]

@dask.delayed
def import_helios_leg(pulse_file, xyz_file):
    # Import all rays into dask dataframe
    leg_rays = dd.read_csv(pulse_file, delimiter=' ', header=None, names=['origin_x', 'origin_y', 'origin_z', 'direction_x', 'direction_y', 'direction_z', 'gps_time', 'ray_id', '_'])
    
    # Format leg_rays
    leg_rays = leg_rays.drop(columns=['gps_time', '_'])

    leg_rays = leg_rays.map_partitions(valid_mask, meta=leg_rays._meta)

    # Import all hits into dask dataframe
    leg_hits = dd.read_csv(xyz_file, delimiter=' ', header=None, names=['points_x', 'points_y', 'points_z', 'intensity', 'echo_width', 'return_number', 'number_of_returns', 'ray_id', 'hit_object_id', 'class', 'gps_time'])
    
    # Format leg_hits. Adjust later for multi-return
    leg_hits = leg_hits.drop(columns=['intensity', 'echo_width', 'return_number', 'number_of_returns', 'hit_object_id', 'class', 'gps_time'])

    leg_rays = leg_rays.merge(leg_hits, on='ray_id', how='left')

    # DEBUG Check for mismatches
    # num_hits = len(leg_hits)
    # num_non_nan_points = (~leg_rays['points_x'].isna()).sum().compute()
    # if num_hits != num_non_nan_points:
    #     print(f"Warning: Mismatch detected! Number of hits: {num_hits}, Number of non-NaN points_x in rays: {num_non_nan_points}")

    # Repartition for optimal performance
    # leg_rays = leg_rays.repartition(npartitions=3)
        
    return leg_rays

# Import all the legs
for leg in selected_legs:
    pulse_file = pulses_template.format(leg=leg)
    xyz_file = xyz_template.format(leg=leg)

    delayed_result = import_helios_leg(pulse_file, xyz_file)
    rays_list.append(delayed_result) 



# Compute the delayed tasks
# Log file to record total number of rays and points
log_file = "rays_points_log.txt"
if os.path.exists(log_file):
    os.remove(log_file)
total_rays = 0
total_points = 0

with ProgressBar():
    # Process each leg in parallel and save to .parquet
    results = dask.compute(*[rays_list[i] for i in range(len(selected_legs))]) #(*[(non_hit_rays_list[i], hit_rays_list[i]) for i in range(len(selected_legs))])

    for i, leg in enumerate(selected_legs):
        print(f"Processing leg {leg}...")

        # Get non-hit rays and hit rays
        rays = results[i]
        
        with open(log_file, "a") as log:
            num_rays = len(rays)
            num_points = (~rays['points_x'].isna()).sum().compute()
            log.write(f"Leg {leg}: Total Rays = {num_rays}, Total Points = {num_points}\n")
        total_rays += num_rays
        total_points += num_points

        # convert leg to int for naming consistency in later computations
        leg = int(leg)
        rays_file = valid_rays_template.format(leg=leg)
        if os.path.exists(rays_file):
            if os.path.isfile(rays_file):
                os.remove(rays_file)
            else:
                shutil.rmtree(rays_file)
        rays.to_parquet(rays_file, engine='pyarrow', compression='snappy', schema=valid_rays_schema)

        # Log aggregated totals
    with open(log_file, "a") as log:
        log.write(f"Aggregated: Total Rays = {total_rays}, Total Points = {total_points}\n")

    # Save all voxel centres to .parquet
    for voxel_size in voxel_sizes:
        voxel_centers = voxels_dict[voxel_size]

        voxel_centers_file = f"200_Leaf_60/voxel_centers_{voxel_size}.parquet"
        if os.path.exists(voxel_centers_file):
            if os.path.isfile(voxel_centers_file):
                os.remove(voxel_centers_file)
            else:
                shutil.rmtree(voxel_centers_file)
        voxel_centers_out = dd.from_array(voxel_centers, columns=['voxel_id', 'voxel_center_x', 'voxel_center_y', 'voxel_center_z', 'voxel_min_x', 'voxel_min_y', 'voxel_min_z', 'voxel_max_x', 'voxel_max_y', 'voxel_max_z'])
        voxel_centers_out.to_parquet(voxel_centers_file, engine='pyarrow', compression='snappy', schema=voxel_centers_schema)
    
print("Done!")

[########################################] | 100% Completed | 327.24 ms
Processing leg 0...
[########################################] | 100% Completed | 2.44 sms
[####                                    ] | 11% Completed | 1.72 s ms

### Importing scan positions from Riegl projects
Use the following section to convert riegl scans into the appropriate format

TBC

## Step 2 - Voxel Ray Intersection
After properly formatting the rays and points, we now need to traverse voxels (of given sizes) and establish the voxel ray intersections. At this stage we want to process all legs in the processed data; the compute metrics stage can simply choose which legs/scan positions to actually use for calculations.

In [1]:

import re
import os
import glob
import pandas as pd
import numpy as np
import pyarrow as pa
import dask.dataframe as dd
from dask.diagnostics import ProgressBar
from dask import delayed, compute

n_missing_points = 0
missing_point_rays = {}

# Establish schemas to avoid conflicts in dask dataframe reading and saving
valid_rays_schema = pa.schema([
    pa.field('ray_id', pa.int64()),
    pa.field('origin_x', pa.float32()),
    pa.field('origin_y', pa.float32()),
    pa.field('origin_z', pa.float32()),
    pa.field('direction_x', pa.float32()),
    pa.field('direction_y', pa.float32()),
    pa.field('direction_z', pa.float32()),
    pa.field('points_x', pa.float32()),
    pa.field('points_y', pa.float32()),
    pa.field('points_z', pa.float32())
])

voxel_centers_schema = pa.schema([
    pa.field('voxel_id', pa.int64()),
    pa.field('voxel_center_x', pa.float32()),
    pa.field('voxel_center_y', pa.float32()),
    pa.field('voxel_center_z', pa.float32()),
    pa.field('voxel_min_x', pa.float32()),
    pa.field('voxel_min_y', pa.float32()),
    pa.field('voxel_min_z', pa.float32()),
    pa.field('voxel_max_x', pa.float32()),
    pa.field('voxel_max_y', pa.float32()),
    pa.field('voxel_max_z', pa.float32())
])

voxel_ray_intersection_schema = pa.schema([
    pa.field('voxel_size', pa.float32()),
    pa.field('voxel_id', pa.int64()),
    pa.field('leg_id', pa.int64()),
    pa.field('ray_id', pa.int64()),
    pa.field('t_entry_x', pa.float32()),
    pa.field('t_entry_y', pa.float32()),
    pa.field('t_entry_z', pa.float32()),
    pa.field('t_exit_x', pa.float32()),
    pa.field('t_exit_y', pa.float32()),
    pa.field('t_exit_z', pa.float32()),
    pa.field('t_entry_radius', pa.float32()),
    pa.field('t_exit_radius', pa.float32()),
    pa.field('ray_points_x', pa.float32()),
    pa.field('ray_points_y', pa.float32()),
    pa.field('ray_points_z', pa.float32()),
    pa.field('viewing_angles', pa.float32()),
    pa.field('hit_ray', pa.bool_())
])

beam_divergence = 0.001  # radians

# Establish file paths
raw_data_path = "200_Leaf_60"
voxel_ray_intersection_path = os.path.join(raw_data_path, "voxel_ray_intersections")
if not os.path.exists(voxel_ray_intersection_path):
    os.makedirs(voxel_ray_intersection_path, exist_ok=True)

leg_patterns = glob.glob(os.path.join(raw_data_path, "leg_*_valid_rays.parquet"))

dfs = []
for leg_pattern in leg_patterns:
    leg = int(re.search(r"leg_(\d+)_valid_rays.parquet", leg_pattern).group(1))
    df = dd.read_parquet(leg_pattern, engine='pyarrow', columns=['ray_id', 'origin_x', 'origin_y', 'origin_z', 'direction_x', 'direction_y', 'direction_z', 'points_x', 'points_y', 'points_z'])
    df['leg_id'] = leg
    dfs.append(df)

legs_df = dd.concat(dfs).compute()

voxel_patterns = glob.glob(os.path.join(raw_data_path, "voxel_centers_*.parquet"))

dfs = []
for voxel_pattern in voxel_patterns:
    voxel_size = float(re.search(r"voxel_centers_([\d.]+).parquet", voxel_pattern).group(1))
    df = dd.read_parquet(voxel_pattern, engine='pyarrow', columns=['voxel_id', 'voxel_center_x', 'voxel_center_y', 'voxel_center_z', 'voxel_min_x', 'voxel_min_y', 'voxel_min_z', 'voxel_max_x', 'voxel_max_y', 'voxel_max_z'])
    df['voxel_size'] = voxel_size
    dfs.append(df)

voxel_sizes_df = dd.concat(dfs).compute()


# Calculate viewing angles from direction vectors
def find_viewing_angles(directions, reference_vector=np.array([0, 0, 1])):
    dir_norms = np.linalg.norm(directions, axis=1, keepdims=True)
    normalized_directions = directions / dir_norms
    dot_products = np.dot(normalized_directions, reference_vector)
    cos_thetas = np.clip(dot_products, -1, 1)
    viewing_angles = np.arccos(cos_thetas)
    return viewing_angles

# Traverse voxels
def traverse_voxels(voxel_partition, leg_id, ray_ids, origins, directions, points, viewing_angles, tolerance=1e-6):
    if voxel_partition.empty:
        return pd.DataFrame(columns=voxel_ray_intersection_schema.names)

    voxel_id = voxel_partition['voxel_id'].values
    voxel_size = voxel_partition['voxel_size'].values
    voxel_min = voxel_partition[['voxel_min_x', 'voxel_min_y', 'voxel_min_z']].values
    voxel_max = voxel_partition[['voxel_max_x', 'voxel_max_y', 'voxel_max_z']].values

    voxel_min = np.asarray(voxel_min) - tolerance
    voxel_max = np.asarray(voxel_max) + tolerance

    voxel_min = np.broadcast_to(voxel_min, (origins.shape[0], voxel_min.shape[1]))
    voxel_max = np.broadcast_to(voxel_max, (origins.shape[0], voxel_max.shape[1]))

    t_min = np.divide(voxel_min - origins, directions, out=np.full_like(origins, float(np.inf)), where=directions != 0)
    t_max = np.divide(voxel_max - origins, directions, out=np.full_like(origins, float(np.inf)), where=directions != 0)

    t1 = np.minimum(t_min, t_max)
    t2 = np.maximum(t_min, t_max)
    t_enter = np.max(t1, axis=1)
    t_exit = np.min(t2, axis=1)
    mask = (t_enter <= t_exit + tolerance) & (t_exit >= -tolerance)

    if not mask.any():
        return pd.DataFrame(columns=voxel_ray_intersection_schema.names)

    points = points[mask]
    boundary_hit_rays = np.any((points == voxel_min[mask]) | (points == voxel_max[mask]), axis=1)
    filtered_hit_rays = np.all((points >= voxel_min[mask]) & (points <= voxel_max[mask]), axis=1) | boundary_hit_rays
    filtered_points_x = np.where(filtered_hit_rays, points[:, 0], np.nan)
    filtered_points_y = np.where(filtered_hit_rays, points[:, 1], np.nan)
    filtered_points_z = np.where(filtered_hit_rays, points[:, 2], np.nan)

    # Check if filtered_points is equal to only true values of hit_rays
    if filtered_hit_rays.sum() != np.count_nonzero(~np.isnan(filtered_points_x)):
        n_hits = filtered_hit_rays.sum()
        n_points = np.count_nonzero(~np.isnan(filtered_points_x))
        point_values = points[filtered_hit_rays]
        point_x_values = point_values[:, 0]
        print("Warning: Mismatch detected! Number of hits and non-NaN points do not match")
        print("Number of hits: ", n_hits)
        print("Number of non-NaN points_x in rays: ", n_points)
        print("First 10 points: ", point_values[:10])
        print("First 10 points_x: ", point_x_values[:10])
        print("Missing points: ", n_hits - n_points)
        print("Ray_ID of missing points: ", ray_ids[filtered_hit_rays]) 
        missing_point_mask = np.isnan(points[filtered_hit_rays]).any(axis=1)
        missing_point_rays[leg_id] = ray_ids[missing_point_mask]
        n_missing_points += n_hits - n_points


    filtered_voxel_size = np.broadcast_to(voxel_size, (mask.sum(),))
    filtered_voxel_id = np.broadcast_to(voxel_id, (mask.sum(),))
    filtered_ray_ids = ray_ids[mask]
    filtered_leg_id = np.broadcast_to(leg_id, (mask.sum(),))
    filtered_viewing_angles = viewing_angles[mask]

    filtered_origins = origins[mask]
    filtered_directions = directions[mask]
    filtered_t_enter = t_enter[mask]
    filtered_t_exit = t_exit[mask]

    t_entry_coords = filtered_origins + filtered_t_enter[:, None] * filtered_directions
    t_exit_coords = filtered_origins + filtered_t_exit[:, None] * filtered_directions

    distance_to_entry = np.linalg.norm(t_entry_coords - filtered_origins, axis=1).astype(np.float32)
    t_entry_radius = (distance_to_entry * np.tan(beam_divergence)).astype(np.float32)
    distance_to_exit = np.linalg.norm(t_exit_coords - filtered_origins, axis=1).astype(np.float32)
    t_exit_radius = (distance_to_exit * np.tan(beam_divergence)).astype(np.float32)

    del filtered_origins, filtered_directions, filtered_t_enter, filtered_t_exit, distance_to_entry, distance_to_exit

    result = pd.DataFrame({
        'voxel_size': filtered_voxel_size,
        'voxel_id': filtered_voxel_id,
        'leg_id': filtered_leg_id,
        'ray_id': filtered_ray_ids,
        't_entry_x': t_entry_coords[:, 0],
        't_entry_y': t_entry_coords[:, 1],
        't_entry_z': t_entry_coords[:, 2],
        't_exit_x': t_exit_coords[:, 0],
        't_exit_y': t_exit_coords[:, 1],
        't_exit_z': t_exit_coords[:, 2],
        't_entry_radius': t_entry_radius,
        't_exit_radius': t_exit_radius,
        'ray_points_x': filtered_points_x,
        'ray_points_y': filtered_points_y,
        'ray_points_z': filtered_points_z,
        'viewing_angles': filtered_viewing_angles,
        'hit_ray': filtered_hit_rays
    })

    return result

# Save to Parquet
def save_to_parquet(df, leg_id, voxel_size):
    if df.empty:
        return

    output_path = os.path.join(voxel_ray_intersection_path, f"leg_{leg_id}_voxel_{voxel_size}_ray_intersections.parquet")
    df.to_parquet(output_path, engine='pyarrow', compression='snappy', schema=voxel_ray_intersection_schema)

# Process each leg and voxel size

delayed_results = []

def process_leg(leg_id, voxel_size):
    leg_df = legs_df[legs_df['leg_id'] == leg_id]
    ray_ids = leg_df['ray_id'].values
    origins = np.stack([leg_df['origin_x'].values, leg_df['origin_y'].values, leg_df['origin_z'].values], axis=1)
    directions = np.stack([leg_df['direction_x'].values, leg_df['direction_y'].values, leg_df['direction_z'].values], axis=1)
    points = np.stack([leg_df['points_x'].values, leg_df['points_y'].values, leg_df['points_z'].values], axis=1)
    viewing_angles = find_viewing_angles(directions)

    results = []

    for _, voxel_partition in voxel_df.groupby(['voxel_size', 'voxel_id']):
        result = traverse_voxels(voxel_partition, leg_id, ray_ids, origins, directions, points, viewing_angles)
        results.append(result)

    if results:
        combined_results = pd.concat(results)
        save_to_parquet(combined_results, leg_id, voxel_size)

for voxel_size in voxel_sizes_df['voxel_size'].unique():
    voxel_df = voxel_sizes_df[voxel_sizes_df['voxel_size'] == voxel_size]
    for leg_id in legs_df['leg_id'].unique():
        delayed_result = delayed(process_leg)(leg_id, voxel_size)
        delayed_results.append(delayed_result)

for result in delayed_results:
    with ProgressBar():
        compute(result)

print("Done!")

if missing_point_rays:
    missing_points_df = pd.DataFrame.from_dict(missing_point_rays, orient='index').transpose()
    missing_points_df.to_csv(os.path.join(raw_data_path, "missing_point_rays.csv"), index=False)

[########################################] | 100% Completed | 28.89 s
Done!
[########################################] | 100% Completed | 27.57 s
Done!
[########################################] | 100% Completed | 28.88 s
Done!
[########################################] | 100% Completed | 28.18 s
Done!
[########################################] | 100% Completed | 29.39 s
Done!
[########################################] | 100% Completed | 27.64 s
Done!
[########################################] | 100% Completed | 29.90 s
Done!
[########################################] | 100% Completed | 28.58 s
Done!
[########################################] | 100% Completed | 30.21 s
Done!
[########################################] | 100% Completed | 27.95 s
Done!
[########################################] | 100% Completed | 28.26 s
Done!
[########################################] | 100% Completed | 28.41 s
Done!


## Step 3 - Compute Metrics
This last code block is used to calculate metrics. Since all the rays have been imported and processed within the leg_{leg}_voxel_{voxel_size}_ray_intersections.parquet files, you simply choose the voxel_sizes and legs you want to calculate metrics for (i.e. select every second leg for sparser sampling distribution etc.)

In [2]:
import os
import open3d as o3d
import numpy as np
import dask.dataframe as dd
import dask
import dask.array as da
import pandas as pd
from sklearn.neighbors import NearestNeighbors
import shutil

voxel_centers_schema = {
    'voxel_id': np.int64(),
    'voxel_center_x': np.float32(),
    'voxel_center_y': np.float32(),
    'voxel_center_z': np.float32(),
    'voxel_min_x': np.float32(),
    'voxel_min_y': np.float32(),
    'voxel_min_z': np.float32(),
    'voxel_max_x': np.float32(),
    'voxel_max_y': np.float32(),
    'voxel_max_z': np.float32()
}

voxel_metrics_schema = {
    'voxel_center_x': np.float32,
    'voxel_center_y': np.float32,
    'voxel_center_z': np.float32,
    'num_hit_rays': np.int64,
    'num_non_hit_rays': np.int64,
    'total_number_rays': np.int64,
    'number_of_points': np.int64,
    'pgap': np.float64,
    'I': np.float64,
    'proceed_with_lad': np.bool_,
    'mean_path_length': np.float64,
    'sum_path_length': np.float64,
    'mean_free_path_length': np.float64,
    'sum_free_path_length': np.float64,
    'mean_delta_e': np.float64,
    'sum_z_e': np.float64,
    'var_delta_e': np.float64,
    'sum_hits_z_e': np.float64,
    'G_mean': np.float64,
    'LAD_BeerLambert': np.float64,
    'LAD_MLE_MCF': np.float64,
    'LAD_BiasCorrectedBeerLambert': np.float64,
    'LAD_BiasCorrectedUnequalPathLengths': np.float64,
    'LAD_TBC_MLE': np.float64,
    'LAD_RCT': np.float64,
    'LAD_RCT_TIM': np.float64,
    'mean_leaf_angle': np.float64
}

def calculate_voxel_metrics(voxel_df, min_rays=6):
    # print("Calculating voxel metrics for voxel_id:", voxel_df['voxel_id'].iloc[0])
    N = voxel_df['ray_id'].count()
    n_hits = voxel_df['hit_ray'].sum()
    # print(f"Total rays: {N}, Hit rays: {n_hits}")

    # print(voxel_df.head())

    # Calculate pgap and I
    if N > 0:
        pgap = (N - n_hits) / N
        I = n_hits / N
    else:
        pgap = np.nan
        I = np.nan
    # print(f"pgap: {pgap}, I: {I}")

    # Calculate path lengths
    path_lengths = np.linalg.norm(voxel_df[['t_exit_x', 't_exit_y', 't_exit_z']].values - voxel_df[['t_entry_x', 't_entry_y', 't_entry_z']].values, axis=1)
    free_path_lengths = np.where(
        voxel_df['hit_ray'], 
        np.linalg.norm(voxel_df[['ray_points_x', 'ray_points_y', 'ray_points_z']].values - voxel_df[['t_entry_x', 't_entry_y', 't_entry_z']].values, axis=1), 
        path_lengths
    )
    # print(f"Path lengths: {path_lengths}, Free path lengths: {free_path_lengths}")

    # Calculate sums and means
    sum_delta = np.sum(path_lengths) if path_lengths.size > 0 else 0
    delta_bar = np.mean(path_lengths) if path_lengths.size > 0 else np.nan
    sum_z = np.sum(free_path_lengths) if free_path_lengths.size > 0 else 0
    z_bar = np.mean(free_path_lengths) if free_path_lengths.size > 0 else np.nan
    # print(f"Sum delta: {sum_delta}, Delta bar: {delta_bar}, Sum z: {sum_z}, Z bar: {z_bar}")

    # Effective path lengths calculation based on delta (path lengths inside the voxel)
    with np.errstate(divide='ignore', invalid='ignore'):
        # Avoid log of zero or negative numbers
        valid_mask_delta = (lambda_1 * path_lengths) < 1
        delta_e = np.full_like(path_lengths, fill_value=np.nan, dtype=np.float64)
        delta_e[valid_mask_delta] = -np.log(1 - lambda_1 * path_lengths[valid_mask_delta]) / lambda_1
    # print(f"Delta effective: {delta_e}")

    mean_delta_e = np.nanmean(delta_e) if not np.isnan(np.nanmean(delta_e)) else 0
    var_delta_e = np.nanvar(delta_e, ddof=1) if np.sum(~np.isnan(delta_e)) > 1 else np.nan
    # print(f"Mean delta effective: {mean_delta_e}, Variance delta effective: {var_delta_e}")

    # Effective Free Path length based on z
    ze = effective_free_path(free_path_lengths, lambda_1)  # Compute effective free path lengths
    # print(f"Effective free path lengths: {ze}")

    viewing_angles = voxel_df['viewing_angles'].values

    # Retrieve LIAD data and viewing angles
    # Filter out NaN points
    valid_points = voxel_df[['ray_points_x', 'ray_points_y', 'ray_points_z']].values
    valid_points = valid_points[~np.isnan(valid_points).any(axis=1)]
    num_valid_points = valid_points.shape[0]
    # print(f"Valid points: {valid_points}, Number of valid points: {num_valid_points}")

    if num_valid_points != n_hits:
        print(f"Voxel {voxel_df['voxel_id'].iloc[0]} has {n_hits} hits but {num_valid_points} valid points.")
    
    leaf_bin_centers, LIAD_values, leaf_angles = calculate_LIAD(valid_points)
    # print(f"Leaf bin centers: {leaf_bin_centers}, LIAD values: {LIAD_values}")

    G_mean = calculate_G_mean(viewing_angles, leaf_bin_centers, LIAD_values)
    invalid_ze_mask = ze < 0
    if np.any(invalid_ze_mask):
        print(f"Voxel {voxel_df['voxel_id'].iloc[0]} has negative ze values. Setting them to zero.")
        ze[invalid_ze_mask] = 0.0

    sum_z_e = np.sum(ze[np.isfinite(ze)]) if ze.size > 0 else 0
    sum_hits_z_e = np.sum(ze[:n_hits]) if n_hits > 0 else 0,
    # print(f"Sum z effective: {sum_z_e}")

    if n_hits > 0 and len(viewing_angles) > 0 and len(leaf_bin_centers) > 0 and len(LIAD_values) > 0 and N >= min_rays:
        if np.isnan(G_mean):
            print(f"Voxel {voxel_df['voxel_id'].iloc[0]} has NaN G_mean.")
    else:
        G_mean = np.nan
        if n_hits > 0:
            print(f"Voxel {voxel_df['voxel_id'].iloc[0]} has hits but insufficient data for G_mean.")

    # Calculate LAD Metrics
    proceed_with_lad = 0 < pgap < 1
    lad = calculate_lad_metrics(G_mean, N, n_hits, delta_bar, sum_z, sum_z_e, sum_hits_z_e, mean_delta_e, var_delta_e, pgap, I, sum_delta, proceed_with_lad)
    mean_leaf_angle = np.mean(leaf_angles) if len(leaf_angles) > 0 else np.nan

    # Assign metrics
    voxel_metrics = pd.Series({
        'num_hit_rays': np.int64(n_hits),
        'num_non_hit_rays': np.int64(N - n_hits),
        'total_number_rays': np.int64(N),
        'number_of_points': np.int64(num_valid_points),
        'pgap': np.float64(pgap),
        'I': np.float64(I),
        'proceed_with_lad': np.bool_(0 < pgap < 1),
        'mean_path_length': np.float64(delta_bar),
        'sum_path_length': np.float64(sum_delta),
        'mean_free_path_length': np.float64(z_bar),
        'sum_free_path_length': np.float64(sum_z),
        'mean_delta_e': np.float64(mean_delta_e),
        'sum_z_e': np.float64(sum_z_e),
        'var_delta_e': np.float64(var_delta_e),
        'sum_hits_z_e': np.float64(np.sum(ze[:n_hits]) if n_hits > 0 else 0),
        'G_mean': np.float64(G_mean),
        'LAD_BeerLambert': lad.get('LAD_BeerLambert', np.nan),
        'LAD_MLE_MCF': lad.get('LAD_MLE_MCF', np.nan),
        'LAD_BiasCorrectedBeerLambert': lad.get('LAD_BiasCorrectedBeerLambert', np.nan),
        'LAD_BiasCorrectedUnequalPathLengths': lad.get('LAD_BiasCorrectedUnequalPathLengths', np.nan),
        'LAD_TBC_MLE': lad.get('LAD_TBC_MLE', np.nan),
        'LAD_RCT': lad.get('LAD_RCT', np.nan),
        'LAD_RCT_Tim': lad.get('LAD_RCT_Tim', np.nan),
        'mean_leaf_angle': mean_leaf_angle
    })

    # print(f"Voxel metrics for voxel_id {voxel_df['voxel_id'].iloc[0]}: {voxel_metrics}")
    return voxel_metrics

def calculate_lad_metrics(G, N, n_hits, delta_bar, sum_z, sum_z_e, sum_hits_z_e, mean_delta_e, var_delta_e, pgap, I, sum_delta, proceed_with_lad, H=1.0):
    """
    Calculates various LAD estimators for each voxel using G_mean directly from voxel_metrics.

    Parameters:
        voxel_metrics (dict): Dictionary mapping voxel indices to computed metrics, including G_mean.
        H (float): Volume fraction not occupied by wood (default is 1.0).

    Returns:
        dict: Dictionary mapping voxel indices to LAD estimations.
    """
    if not proceed_with_lad:
        # Skip LAD calculations
        return {
            'LAD_BeerLambert': np.nan,
            'LAD_MLE_MCF': np.nan,
            'LAD_BiasCorrectedBeerLambert': np.nan,
            'LAD_BiasCorrectedUnequalPathLengths': np.nan,
            'LAD_TBC_MLE': np.nan,
            'LAD_RCT': np.nan,
            'LAD_RCT_Tim': np.nan
        }

    # Handle None or invalid G values
    if G is None or G <= 0:
        logging.warning(f"Voxel {voxel_idx} has G_mean=None or G_mean <= 0: G_mean={G}")
        G = np.nan
    if (delta_bar <= 0):
        logging.warning(f"Voxel {voxel_idx} has delta_bar <= 0: delta_bar={delta_bar}")

    # 1. Beer-Lambert Estimator
    if G > 0 and 0 < pgap < 1 and delta_bar > 0:
        I = 1 - pgap  # Relative density index (RDI)
        LAD_BeerLambert = -np.log(1 - I) / (G * H * delta_bar)
    else:
        LAD_BeerLambert = np.nan

    # 2. Modified Contact Frequency (MCF) / MLE
    if G > 0 and sum_z > 0:
        LAD_MLE_MCF = n_hits / (G * H * sum_z)
    else:
        LAD_MLE_MCF = np.nan

    # 3. Bias-Corrected Beer-Lambert Estimator
    if G > 0 and mean_delta_e > 0 and N > 0 and 0 < pgap < 1:
        I = 1 - pgap  # Relative density index (RDI)
        Lambda_hat = (-np.log(1 - I) / mean_delta_e) * (1 + (1 - I) / (2 * N * I))
        LAD_BiasCorrectedBeerLambert = Lambda_hat / (G * H)
    else:
        LAD_BiasCorrectedBeerLambert = np.nan

    # 4. Bias-Corrected Beer-Lambert Estimator for Unequal Path Lengths
    if G > 0 and mean_delta_e > 0 and var_delta_e > 0 and not np.isnan(LAD_BiasCorrectedBeerLambert):
        Lambda_hat = LAD_BiasCorrectedBeerLambert  # Use Lambda_hat from Equation 3
        Lambda_2 = Lambda_hat * (1 + (Lambda_hat * var_delta_e) / (2 * mean_delta_e))
        LAD_BiasCorrectedUnequalPathLengths = Lambda_2 / (G * H)
    else:
        LAD_BiasCorrectedUnequalPathLengths = np.nan

    # 5. Theoretically Bias-Corrected Maximum Likelihood Estimator (TBC-MLE)
    if G > 0 and sum_z_e > 0 and N > 0:
        correction_term = (sum_hits_z_e / sum_z_e) / (2 * N)
        LAD_TBC_MLE = (n_hits / (G * H * sum_z_e)) * (1 - correction_term)
    else:
        LAD_TBC_MLE = np.nan

    # 6. LAD (Ray Cloud Tools)
    if G > 0 and sum_z > 0 and n_hits > 0:
        lambda_hat = n_hits / sum_z
        rho_c = 2 * G * lambda_hat * (1 - np.exp(-lambda_hat / 2))
    else:
        rho_c = np.nan

    # 7. LAD (Ray Cloud Tools_Tim)
    if G > 0 and sum_z > 0 and n_hits > 0:
        Ãµ = 1e10
        lambda_hat = 2*(N-1)*n_hits/(Ãµ + N*sum_delta)
        rho_c_Tim = lambda_hat / G 
    else:
        rho_c_Tim = np.nan

    lad_metrics = {}

    lad_metrics = {
        'LAD_BeerLambert': LAD_BeerLambert,
        'LAD_MLE_MCF': LAD_MLE_MCF,
        'LAD_BiasCorrectedBeerLambert': LAD_BiasCorrectedBeerLambert,
        'LAD_BiasCorrectedUnequalPathLengths': LAD_BiasCorrectedUnequalPathLengths,
        'LAD_TBC_MLE': LAD_TBC_MLE,
        'LAD_RCT': rho_c,
        'LAD_RCT_Tim': rho_c_Tim
    }

    return lad_metrics

def calculate_path_lengths(voxel_points, voxel_entries, voxel_exits):
    """
    Computes path lengths for each ray in the voxel.

    Parameters:
        voxel_points (np.array): Points of the rays in the voxel.
        voxel_entries (np.array): Entry points of the rays in the voxel.
        voxel_exits (np.array): Exit points of the rays in the voxel.

    Returns:
        np.array: Path lengths for each ray in the voxel.
    """
    return np.linalg.norm(voxel_exits - voxel_entries, axis=1)

def calculate_LIAD(voxel_points, knn=6, radius=0.1, max_nn=10, num_bins=18):
    """
    Computes weights for each point based on voxel point density and local neighborhood.

    Parameters:
        voxel(da.array) - all points in voxel['ray_points_x', 'ray_points_y', 'ray_points_z']

    Returns:
        dict: Mapping from voxel indices to weights for each point.
    """
    # Compute the point density weights
    if len(voxel_points) < knn:
        weights = np.ones(len(voxel_points))
    else:
        nbrs = NearestNeighbors(n_neighbors=knn).fit(voxel_points)
        distances, _ = nbrs.kneighbors(voxel_points)
        # Inverse of the distance to the k-th neighbor as weight
        weights = 1 / (distances[:, -1] + 1e-6)  # Avoid division by zero

    # Compute the normals
    if len(voxel_points) < max_nn:
        return np.array([]), np.array([]), np.array([])

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(voxel_points)
    pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=radius, max_nn=max_nn))
    normals = np.asarray(pcd.normals)

    # Compute the angle between the normal and the leaf point
    angles = np.arccos(np.dot(normals, np.array([0, 0, 1])))

    # Compute LIAD for each voxel
    if len(angles) == 0 or np.all(np.isnan(angles)):
        # voxel_liad[voxel_idx] = (np.array([]), np.array([]))
        pass
    else:
        if len(weights) == 0:
            weights = np.ones_like(angles)

        # Remove NaN angles and align weights
        valid_mask = ~np.isnan(angles)
        angles_valid = angles[valid_mask]
        weights_valid = weights[valid_mask]

        # Ensure data alignment
        # if len(leaf_point_angles_valid) != len(weights_valid):
        #     logging.error(f"Voxel {voxel_idx}: Mismatch in angles ({len(leaf_point_angles_valid)}) and weights ({len(weights_valid)}) lengths.")
        #     skipped_voxels += 1
        #     voxel_liad[voxel_idx] = (np.array([]), np.array([]))
        #     continue

        # Skip voxel if valid data is insufficient
        if len(angles_valid) == 0:
            # skipped_voxels += 1
            # voxel_liad[voxel_idx] = (np.array([]), np.array([]))
            pass

        # Normalize weights for consistency
        total_weights = np.sum(weights_valid)
        if total_weights > 0:
            weights_valid /= total_weights
        else:
            weights_valid = np.ones_like(angles_valid)

        # Compute histogram with weights
        hist, bin_edges = np.histogram(
            angles_valid, bins=num_bins, range=(0, 90), weights=weights_valid, density=False
        )
        total_weight = np.sum(hist)
        if total_weight > 0:
            LIAD_values = hist / total_weight
        else:
            LIAD_values = np.zeros(num_bins)

        # Compute bin centers
        leaf_bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

        return leaf_bin_centers, LIAD_values, angles_valid

def calculate_G_mean(viewing_angles, leaf_bin_centers, LIAD_values):
    """
    Computes the G function for a single voxel.

    Parameters:
        viewing_angles (np.array): Viewing angles of the rays.
        leaf_bin_centers (np.array): Bin centers for the leaf angle histogram.
        LIAD_values (np.array): Local Intensity-Angle Distribution values.

    Returns:
        float: G function value for the voxel.
    """

    if len(viewing_angles) == 0 or len(leaf_bin_centers) == 0:
        return np.nan

    # Convert degrees to radians
    theta = np.radians(viewing_angles)
    theta_l = np.radians(leaf_bin_centers)

    cos_theta = np.cos(theta)
    cos_theta_l = np.cos(theta_l)

    tan_theta = np.tan(theta)
    epsilon = 1e-6
    tan_theta = np.where(tan_theta == 0, epsilon, tan_theta)
    cot_theta = 1 / tan_theta
    max_cot = 1e3
    cot_theta = np.clip(cot_theta, -max_cot, max_cot)

    tan_theta_l = np.tan(theta_l)
    tan_theta_l = np.where(tan_theta_l == 0, epsilon, tan_theta_l)
    cot_theta_l = 1 / tan_theta_l
    cot_theta_l = np.clip(cot_theta_l, -max_cot, max_cot)

    # Normalize LIAD
    LIAD_normalized = LIAD_values / np.sum(LIAD_values) if np.sum(LIAD_values) > 0 else LIAD_values

    # Calculate G values using the vectorized function
    G_values = compute_G_function_binwise_vectorized(
        cos_theta, cos_theta_l, cot_theta, cot_theta_l, LIAD_normalized, max_cot
    )

    # Handle NaN or Inf values
    G_values = np.nan_to_num(G_values, nan=0.0, posinf=0.0, neginf=0.0)

    # Compute the mean G
    G_mean = np.nanmean(G_values) if len(G_values) > 0 else np.nan

    return G_mean

def compute_G_function_binwise_vectorized(cos_theta, cos_theta_l, cot_theta, cot_theta_l, LIAD_normalized, max_cot):
    # Compute the outer product of cot_theta and cot_theta_l
    cp = cot_theta[:, np.newaxis] * cot_theta_l

    # Compute the outer product of cos_theta and cos_theta_l
    cos_product = cos_theta[:, np.newaxis] * cos_theta_l

    # Calculate A using broadcasting
    psi = np.arccos(np.clip(cp, -1, 1))
    tan_psi = np.tan(psi)
    A = np.where(np.abs(cp) > 1, cos_product, cos_product * (1 + (2.0 / np.pi) * (tan_psi - psi)))

    # Sum over the leaf angles and multiply by LIAD_normalized
    G_values = np.sum(A * LIAD_normalized, axis=1)

    return G_values

def calculate_lambda1(voxel_size, r=0.05):
    """
    Calculates lambda_1 based on leaf radius and voxel size.
    """
    if voxel_size <= 0:
        logging.error(f"Invalid voxel size delta: {voxel_size}. Must be positive.")
        return np.nan
    if r <= 0:
        logging.error(f"Invalid leaf radius r: {r}. Must be positive.")
        return np.nan
    return np.pi * r**2 / voxel_size**3

def effective_free_path(z, lambda_1):
    """
    Calculates the effective free path ze for a given free path z using lambda_1.
    """
    z = z.copy()
    with np.errstate(divide='ignore', invalid='ignore'):
        valid_mask = (lambda_1 * z) < 1
        ze = np.full_like(z, fill_value=np.inf, dtype=np.float64)
        ze[valid_mask] = -np.log(1 - lambda_1 * z[valid_mask]) / lambda_1
    return ze

def compute_G_function_binwise_numba(cos_theta, cos_theta_l, cot_theta, cot_theta_l, LIAD_normalized, max_cot):
    """
    Numba-accelerated function to compute G values for viewing angles.

    Parameters:
        cos_theta (float[:]): Cosines of viewing angles.
        cos_theta_l (float[:]): Cosines of leaf angles.
        cot_theta (float[:]): Cotangents of viewing angles.
        cot_theta_l (float[:]): Cotangents of leaf angles.
        LIAD_normalized (float[:]): Normalized LIAD values.
        max_cot (float): Maximum cotangent value to prevent overflow.

    Returns:
        float[:]: G values corresponding to each viewing angle.
    """
    num_theta = len(cos_theta)
    num_theta_l = len(cos_theta_l)
    G_values = np.zeros(num_theta)
    
    for i in prange(num_theta):
        for j in range(num_theta_l):
            cp = cot_theta[i] * cot_theta_l[j]
            if np.abs(cp) > 1:
                A = cos_theta[i] * cos_theta_l[j]
            else:
                psi = np.arccos(cp)
                tan_psi = np.tan(psi)
                A = cos_theta[i] * cos_theta_l[j] * (1 + (2.0 / np.pi) * (tan_psi - psi))
            G_values[i] += A * LIAD_normalized[j]
    return G_values


# Group pandas dataframe by voxel_id
legs = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]  # Example legs
voxel_sizes = [0.5]  # Example voxel sizes



for size in voxel_sizes:
    voxel_centers_file = f"200_Leaf_60/voxel_centers_{size}.parquet"
    voxel_centers = dd.read_parquet(voxel_centers_file).compute()
    voxel_centers = voxel_centers[['voxel_id', 'voxel_center_x', 'voxel_center_y', 'voxel_center_z']]

    lambda_1 = calculate_lambda1(size)
    print(f"Voxel size: {size}, lambda_1: {lambda_1}")

    # Load and concatenate dataframes
    dataframes = []
    for leg in legs:
        for voxel_size in voxel_sizes:
            file_path = f"200_Leaf_60/voxel_ray_intersections/leg_{leg}_voxel_{voxel_size}_ray_intersections.parquet"
            df = dd.read_parquet(file_path)
            dataframes.append(df)

    combined_df = dd.concat(dataframes).compute()
    voxel_dfs = combined_df.groupby('voxel_id')

    # DEBUG
    # for voxel_id, df in voxel_dfs:
    #     print(df[df['hit_ray'] == True].count())
    #     print(df[df['t_entry_x'] != np.inf].count())
        

    voxel_metrics_dfs = voxel_dfs.apply(calculate_voxel_metrics)
    # print(voxel_metrics_dfs.head())

    voxel_metrics_dfs = voxel_centers.merge(voxel_metrics_dfs, on='voxel_id', how='left')

    output_csv_path = os.path.join("200_Leaf_60", f"voxel_metrics_{size}_legs_{'_'.join(map(str, legs))}.csv")
    if os.path.exists(output_csv_path):
        if os.path.isfile(output_csv_path):
            os.remove(output_csv_path)
        else:
            shutil.rmtree(output_csv_path)
    voxel_metrics_dfs.to_csv(output_csv_path, index=True)
    print(f"Voxel metrics saved to {output_csv_path}")
# Apply the calculate_voxel_metrics function to each group
# voxel_metrics_dfs = voxel_ray_intersections.groupby('voxel_id').apply(calculate_voxel_metrics, meta=)
# print(voxel_metrics_dfs.head())

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Voxel size: 0.5, lambda_1: 0.06283185307179587
Voxel 500000008 has hits but insufficient data for G_mean.
Voxel 500000021 has hits but insufficient data for G_mean.
Voxel 500000025 has hits but insufficient data for G_mean.
Voxel metrics saved to 200_Leaf_60/voxel_metrics_0.5_legs_0_1_2_3_4_5_6_7_8_9_10_11.csv


/tmp/ipykernel_111126/673513706.py:514: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  voxel_metrics_dfs = voxel_dfs.apply(calculate_voxel_metrics)


POINT CHECKER

Create a pointcloud of points which are missing from the final result                   

In [3]:
import dask.dataframe as dd
import open3d as o3d
import numpy as np

legs = [4, 6]
raw_data_path = "200_Leaf_60"
voxel_ray_intersection_path = "200_Leaf_60/voxel_ray_intersections"

total_missing_points = None
total_missing_normals = None

for leg in legs:
    print(f"Processing leg {leg}...")

    # Read the valid rays parquet file
    valid_rays_file = f"{raw_data_path}/leg_{leg}_valid_rays.parquet"
    print(f"Reading valid rays from {valid_rays_file}")
    valid_rays_df = dd.read_parquet(valid_rays_file).compute()

    # Read the leg_i_voxel_0.5 parquet file
    voxel_rays_file = f"{voxel_ray_intersection_path}/leg_{leg}_voxel_0.5_ray_intersections.parquet"
    print(f"Reading voxel rays from {voxel_rays_file}")
    voxel_rays_df = dd.read_parquet(voxel_rays_file).compute()
    voxel_rays_df = voxel_rays_df[~np.isnan(voxel_rays_df['ray_points_x'])]

    # Find the points that are in valid rays but not in voxel rays
    print("Finding missing points...")
    missing_points_df = valid_rays_df[~valid_rays_df[['points_x', 'points_y', 'points_z']].apply(tuple, axis=1).isin(voxel_rays_df[['ray_points_x', 'ray_points_y', 'ray_points_z']].apply(tuple, axis=1))]
    missing_points_df = missing_points_df[~np.isnan(missing_points_df['points_x'])]

    # Create a point cloud of the missing points
    missing_points = missing_points_df[['points_x', 'points_y', 'points_z']].values
    
    print(f"Number of missing points: {len(missing_points)}")
    if len(missing_points) > 0:

        # Calculate normals for the missing points
        origins = missing_points_df[['origin_x', 'origin_y', 'origin_z']].values
        normals = missing_points - origins

        # Add points and normals to totals
        if total_missing_points is None:
            total_missing_points = missing_points
        else:
            total_missing_points = np.concatenate((total_missing_points, missing_points))
        
        if total_missing_normals is None:
            total_missing_normals = normals
        else:
            total_missing_normals = np.concatenate((total_missing_normals, normals))
    
    else:
        print(f"No missing points found for leg {leg}")
    
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(total_missing_points)
pcd.normals = o3d.utility.Vector3dVector(total_missing_normals)
output_file = f"total_missing_points.ply"
print(f"Writing total missing points to {output_file}")
o3d.io.write_point_cloud(output_file, pcd, write_ascii=True)

# Visualise
o3d.visualization.draw_geometries([pcd], point_show_normal=True)
    

Processing leg 4...
Reading valid rays from 200_Leaf_60/leg_4_valid_rays.parquet
Reading voxel rays from 200_Leaf_60/voxel_ray_intersections/leg_4_voxel_0.5_ray_intersections.parquet
Finding missing points...
Number of missing points: 1
Processing leg 6...
Reading valid rays from 200_Leaf_60/leg_6_valid_rays.parquet
Reading voxel rays from 200_Leaf_60/voxel_ray_intersections/leg_6_voxel_0.5_ray_intersections.parquet
Finding missing points...
Number of missing points: 1
Writing total missing points to total_missing_points.ply
[Open3D WARNING] GLFW Error: Wayland: The platform does not support setting the window position
[Open3D WARNING] Failed to initialize GLEW.
[Open3D WARNING] [DrawGeometries] Failed creating OpenGL window.
